# IPPT Predictive Model — Version 2 (Synthetic Data)

**Phase 1 Status: Complete**

**What this notebook does:**
- Generates synthetic (fake) data for 500 "users" with realistic inputs
- Predicts all 3 IPPT components: push-ups, sit-ups, and 2.4km run time
- Trains and compares 2 model types: Linear Regression and Random Forest
- Evaluates each using MAE, RMSE, R², and 5-fold cross-validation
- Shows feature importance (which inputs matter most to the model)
- Includes adherence % as an input, distinguishing between having a
  training plan and actually following it

**Important:** This uses synthetic data only. It proves the pipeline works
end-to-end — it does NOT mean the model can predict real IPPT performance
yet. That requires real data (Phase 2), which is the project's current
blocker.

Run each cell in order (Shift + Enter), or Runtime to Run all.

## Step 0: Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)  # reproducible results

## Step 1: Generate synthetic input features

These are the inputs the model will use. Ranges are chosen to be
realistic for an NSF/NSmen population, but the actual numbers are
randomly generated — not real people.

Note the addition of **adherence %** — this represents how much of the
*planned* training the person actually completed. A user might plan 4
sessions/week but only actually complete 2 of them. Adherence captures
that gap, so the model can reflect that following through on a training
plan matters, not just having one.

In [ ]:
n_samples = 500

age = np.random.randint(18, 40, n_samples)
current_pushups = np.random.randint(10, 60, n_samples)
current_situps = np.random.randint(20, 70, n_samples)
current_runtime = np.random.randint(600, 900, n_samples)  # in seconds (10:00 - 15:00)
training_frequency = np.random.randint(0, 7, n_samples)   # sessions PLANNED per week
training_duration = np.random.randint(0, 13, n_samples)   # weeks trained
adherence_pct = np.random.randint(30, 101, n_samples)     # % of planned sessions actually completed

## Step 2: Define the "ground truth" relationship

This is YOU deciding how the inputs affect future performance. In real
life we don't know this formula — that's what the model has to learn.
Since this is synthetic data, we control it, so afterwards we can check
whether the model correctly rediscovers something close to it.

Note the run time formula works in reverse — training should REDUCE time,
not increase it.

**Adherence is applied as a multiplier on the training effect.** For
example, someone who planned 4 sessions/week for 6 weeks but only
completed 50% of them effectively only gets half the training benefit
in this formula. This is what lets us show, later, that "following the
AI Coach's plan" matters — not just having a good plan on paper.

In [ ]:
# Adherence scales how much of the "planned" training effect a person
# actually gets. 100% adherence = full effect, 50% adherence = half.
adherence_multiplier = adherence_pct / 100

# --- Push-ups: training increases them, diminishing returns, age slows gains
future_pushups = (
    current_pushups
    + training_frequency * training_duration * 0.30 * adherence_multiplier
    - 0.05 * current_pushups
    - 0.10 * (age - 18)
    + np.random.normal(0, 3, n_samples)
)
future_pushups = np.clip(future_pushups, 0, None)

# --- Sit-ups: similar pattern to push-ups, slightly different scale
future_situps = (
    current_situps
    + training_frequency * training_duration * 0.25 * adherence_multiplier
    - 0.04 * current_situps
    - 0.08 * (age - 18)
    + np.random.normal(0, 3, n_samples)
)
future_situps = np.clip(future_situps, 0, None)

# --- Run time (seconds): training REDUCES time, so effects are negative
future_runtime = (
    current_runtime
    - training_frequency * training_duration * 1.2 * adherence_multiplier
    + 0.5 * (age - 18)
    + np.random.normal(0, 10, n_samples)
)
future_runtime = np.clip(future_runtime, 300, None)  # can't go below 5:00

## Step 3: Combine into a single DataFrame

In [ ]:
df = pd.DataFrame({
    "age": age,
    "current_pushups": current_pushups,
    "current_situps": current_situps,
    "current_runtime": current_runtime,
    "training_frequency": training_frequency,
    "training_duration": training_duration,
    "adherence_pct": adherence_pct,
    "future_pushups": future_pushups,
    "future_situps": future_situps,
    "future_runtime": future_runtime,
})

df.head()

## Step 4: Set up features (X) and the 3 targets (y)

We'll train **3 separate models**, one per IPPT component. This mirrors
how IPPT itself is structured (3 stations), and makes it easy to see
which component the model predicts best.

In [ ]:
feature_cols = ["age", "current_pushups", "current_situps",
                "current_runtime", "training_frequency", "training_duration",
                "adherence_pct"]

X = df[feature_cols]
targets = {
    "pushups": df["future_pushups"],
    "situps": df["future_situps"],
    "runtime": df["future_runtime"],
}

## Step 5: Train + evaluate a model for each target

In [ ]:
models = {}
results = {}

for name, y in targets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)
    models[name] = model

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    cv_mae = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")

    results[name] = {
        "MAE": mae, "RMSE": rmse, "R2": r2,
        "CV_MAE_mean": cv_mae.mean(), "CV_MAE_per_fold": cv_mae
    }

    print(f"=== {name.upper()} ===")
    print(f"MAE:  {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R^2:  {r2:.3f}")
    print(f"5-fold CV MAE: {cv_mae.mean():.2f}  (per fold: {np.round(cv_mae, 2)})")
    print()

## Step 6: Check what each model learned

Compare these coefficients to the "ground truth" formulas in Step 2 —
a good model should roughly rediscover them. This is a useful sanity
check to show that the model is learning real structure, not noise.

In [ ]:
for name, model in models.items():
    print(f"=== {name.upper()} coefficients ===")
    for feature, coef in zip(feature_cols, model.coef_):
        print(f"  {feature}: {coef:.3f}")
    print(f"  intercept: {model.intercept_:.3f}")
    print()

## Step 7: Try a prediction for a hypothetical user

In [ ]:
example_user = pd.DataFrame({
    "age": [22],
    "current_pushups": [30],
    "current_situps": [35],
    "current_runtime": [720],  # 12:00
    "training_frequency": [4],
    "training_duration": [6],
    "adherence_pct": [90],
})

pred_pushups = models["pushups"].predict(example_user)[0]
pred_situps = models["situps"].predict(example_user)[0]
pred_runtime = models["runtime"].predict(example_user)[0]

mins, secs = divmod(int(pred_runtime), 60)

print("A 22-year-old with 30 push-ups, 35 sit-ups, 12:00 run,")
print("PLANNING 4x/week for 6 weeks, at 90% adherence, is predicted to reach:")
print(f"  Push-ups: {pred_pushups:.1f}")
print(f"  Sit-ups:  {pred_situps:.1f}")
print(f"  Run time: {mins}:{secs:02d}")

### Same plan, but with poor adherence

Same person, same 4x/week plan for 6 weeks — but this time they only
actually complete about half their planned sessions (50% adherence
instead of 90%). This shows how much having a good plan matters less
if it isn't actually followed.

In [ ]:
example_user_low_adherence = example_user.copy()
example_user_low_adherence["adherence_pct"] = [50]

pred_pushups_low = models["pushups"].predict(example_user_low_adherence)[0]
pred_situps_low = models["situps"].predict(example_user_low_adherence)[0]
pred_runtime_low = models["runtime"].predict(example_user_low_adherence)[0]

mins_low, secs_low = divmod(int(pred_runtime_low), 60)

print("Same plan, but only 50% adherence:")
print(f"  Push-ups: {pred_pushups_low:.1f}  (vs {pred_pushups:.1f} at 90% adherence)")
print(f"  Sit-ups:  {pred_situps_low:.1f}  (vs {pred_situps:.1f} at 90% adherence)")
print(f"  Run time: {mins_low}:{secs_low:02d}  (vs {mins}:{secs:02d} at 90% adherence)")

## Step 8: Try a second model type — Random Forest

So far we've only used **Linear Regression**, which assumes a straight-line
relationship between inputs and outputs.

**Random Forest** works differently — it builds many simple decision trees
(each one asking a series of yes/no questions about the inputs, e.g. "is
training frequency > 3?") and averages their predictions together. It can
pick up on patterns that aren't simple straight lines.

We're not replacing the Linear Regression models — we're training a second
model per target and comparing which does better. Comparing multiple model
types like this is standard practice before deciding which one to use.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_models = {}
rf_results = {}

for name, y in targets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_models[name] = rf_model

    y_pred = rf_model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    cv_mae = -cross_val_score(rf_model, X, y, cv=5, scoring="neg_mean_absolute_error")

    rf_results[name] = {
        "MAE": mae, "RMSE": rmse, "R2": r2, "CV_MAE_mean": cv_mae.mean()
    }

    print(f"=== {name.upper()} (Random Forest) ===")
    print(f"MAE:  {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R^2:  {r2:.3f}")
    print(f"5-fold CV MAE: {cv_mae.mean():.2f}")
    print()

## Step 9: Compare Linear Regression vs Random Forest

This table puts both models side by side so you can see, for each IPPT
component, which one performed better (lower MAE/RMSE = better,
higher R² = better).

In [ ]:
comparison_rows = []
for name in targets.keys():
    comparison_rows.append({
        "target": name,
        "LinearRegression_MAE": results[name]["MAE"],
        "RandomForest_MAE": rf_results[name]["MAE"],
        "LinearRegression_R2": results[name]["R2"],
        "RandomForest_R2": rf_results[name]["R2"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

**How to read this table:** for each row (push-ups, sit-ups, run time),
compare the two MAE columns — whichever is lower did better on that
target. It's normal for neither model to win on everything. On clean
synthetic data like this, Linear Regression often does fine since we
built the data using fairly linear rules — Random Forest tends to show
its advantage more on messier, real-world data with more complex
patterns.

## Step 10: Feature importance (Random Forest)

One advantage of Random Forest is that it can tell us which inputs it
relied on most to make its predictions. This is useful for explaining
the model to non-technical people (e.g. commanders) — you can say
"the model found X mattered most for Y" instead of just showing
error numbers.

Higher importance = the model used that feature more when making
its decisions. These are relative — they add up to 1.0 (100%) across
all features for a given target.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name in zip(axes, targets.keys()):
    importances = rf_models[name].feature_importances_
    imp_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": importances
    }).sort_values("importance", ascending=True)

    ax.barh(imp_df["feature"], imp_df["importance"])
    ax.set_title(f"Feature importance: {name}")
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.show()

**How to read this:** the longest bars are the features the model
relied on most. Given how we built the synthetic data (Step 2), you'd
expect `training_frequency` and `training_duration` to show up as
important for push-ups and sit-ups, and less so for things like
`current_situps` when predicting `future_pushups`. If the chart roughly
matches that expectation, it's another sign the model learned sensible
patterns rather than noise.

Once real data is used, this chart becomes genuinely useful — it could
reveal, for example, that adherence matters more than raw training
frequency, which would directly inform how the AI Coach should be
designed.

## Notes / Next steps

- This is still synthetic data — the numbers above don't represent real
  IPPT predictions, only that the pipeline (data → train → evaluate →
  predict) works end-to-end for all 3 components.
- Next step per the roadmap: obtain real-world data (Phase 2) before
  these models can be validated or trusted.
- When real data is available, only Steps 1–3 (data generation) need to
  change — load a real CSV instead. Steps 4 onward stay the same.
- We've now compared 2 model types (Linear Regression, Random Forest).
  This directly feeds into Phase 3 of the roadmap, which calls for
  training and comparing multiple models once real data is available.
- Feature importance gives a first look at which inputs matter most —
  a real-data version of this chart could help shape what the AI Coach
  actually focuses on.
- We added **adherence %** as an input — the model now distinguishes
  between having a good training plan and actually following it. This
  matches a real gap identified in the data schema (Phase 2 groundwork):
  the app needs to log actual completed sessions, not just planned ones.